In [1]:
import numpy as np
import polars as pl
import pandas as pd
import os
import sys
from pathlib import Path
import duckdb
import datetime as dt
parent = Path.cwd()
while 'notebooks' in str(parent):
    parent = parent.parent

sys.path.insert(1, str(parent))


In [8]:
from src_strategy.configs.dataconfig import data_config, input_output_config_4, bp_config
from src_strategy.data_ingest.dataloader_2 import DataLoader
from src_strategy.configs.outlierdetection.extremeoutliers import flowsheet_bounds_config, lab_bounds_config, bp_bounds_config

In [9]:
dl = DataLoader(input_output_config_4,
                data_config,
                bp_config,
                flowsheet_bounds_config,
                lab_bounds_config,
                bp_bounds_config)

In [13]:
sources = dl._read_sources()

In [15]:
sources = dl._deduplicate_sources(sources)
sources = dl._transform_sources(sources)

In [16]:
if 'flowsheets' not in sources:
	sources['flowsheets'] = pl.concat([
		sources['flowsheets_other'].select(sources['flowsheets_sirs'].columns), sources['flowsheets_sirs']
	], how='vertical')

	sources['flowsheets'] = sources['flowsheets'].join(
		sources['flowsheets_other'].select(
			[
				'EncounterEpicCsn', 'Event_DateTime', 'Event_ID',
				'Type', 'Event_Grouper', 'Event_Name', 'sys', 'map', 'dia',
				'sys_is_outlier', 'map_is_outlier', 'dia_is_outlier', 'flowsheet_outlier'
			]
		),
		on=['EncounterEpicCsn', 'Event_DateTime', 'Event_ID', 'Event_Name'],
		how='left',
		suffix='_other'
	)

for c in sources['flowsheets'].columns:
	if not c.endswith('_other'):
		continue
	if sources['flowsheets'].filter(
		pl.col(c) != pl.col(c.split('_other')[0])
	).shape[0] > 0:
		raise ValueError(f"flowsheets columns {c} and {c.split('_other')[0]} do not match ")
del sources['flowsheets_other']
del sources['flowsheets_sirs']
sources['flowsheets'] = sources['flowsheets'].drop([c for c in sources['flowsheets'].columns if c.endswith('_other')])

sources = dl._audit_and_clean_sources(sources)
# sources, encounters = dl._finalize_source_columns(sources)

In [25]:
source_frames = []

for name in dl.TRANSACTIONAL_SOURCES:
	source_family = (
		"flowsheets" if name in  ('flowsheets_other', 'flowsheets_sirs') else name
	)

	source_frames.append(
		sources[source_family].select(data_config.base_cols).with_columns(pl.lit(source_family).alias('_source_family'))
	)

source_events = pl.concat(source_frames, how='vertical' )

In [26]:
type_source_check = (
    source_events.group_by("Type").agg(
        pl.col("_source_family").n_unique().alias("source_family_count")
	)
	.filter(pl.col("source_family_count") > 1)
)

In [27]:
type_source_check

Type,source_family_count
str,u64


In [31]:
source_events['Event_ID'].null_count()

0